In [1]:
# Imports
import time
import asyncio
from typing import Literal
from enum import Enum, auto

# Dependencies
import pandas as pd

# Library
from fsm import	fsm, menus
from fsm.states import State
from fsm.menus import NATIONS
from utils.utils import construct_rules
from fsm.routines.parse import parse_links
from fsm.routines.fetch import fetch_routine
from pipeline.builder import VanguardPipeline
from pipeline.query_builder import make_query
from data.vanguard_data import VanguardStorage
from fsm.routines.scrap import main_scrap_routine
from parsers.vanguard_parser import VanguardParser
from classifier.vanguard_classifier import VanguardClassifier
from api_builder.vanguard_api_build import MediaWikiAPI, VanguardScrapper
from api_builder.api_request import header


In [2]:
# Url Parser
web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardScrapper(web),
	VanguardParser(),
	VanguardClassifier(),
	VanguardStorage()
)
await pipeline.scrapper.api.init_session()
state_machine = fsm.FSMContext()
state = State.ENTRY_POINT
while (state != State.END):
	if (state == State.ENTRY_POINT):
		state = menus.entry_point(state_machine)
	elif (state == State.SELECT_MAIN_CATEGORY):
		state = menus.select_category(state_machine)
	elif (state == State.SELECT_SUBCATEGORY):
		state = menus.select_subcategory(state_machine)
	elif (state == State.BUILD_QUERY):
		state = make_query(state_machine)
	elif (state == State.FETCH):
		state = await fetch_routine(state_machine, pipeline.scrapper)
	elif (state == State.PARSE):
		pipeline.classifier._define_rules(construct_rules(state_machine.main_category.capitalize()))
		state = parse_links(
			state_machine, pipeline.parser, pipeline.storage,
			pipeline.scrapper, pipeline.classifier
		)
time.sleep(4)

Welcome to VanguardTCGScrapper


What info do you need from the website?
0 : boosters
1 : specials
2 : decks
3 : others
You selected: boosters
Which subcategory you want to scrap?
indentify it with the index number:

0:  Booster Sets
1:  Extra Booster Sets
2:  Character Booster Sets
3:  Clan Booster Sets
4:  Title Booster Sets
5:  Unique Booster Sets


In [ ]:
from fsm.routines.check_data_base import build_set_path

for block in ["LB", "LL", "G", "V", "D", "DZ"]:
	consult = pipeline.parser.make_consults(getattr(pipeline.storage, block.lower()))
	print(consult)
	for tpl in consult.values():
		print(f"voy a pasar esta consulta: {tpl}")
		time.sleep(6)
		api_result = await pipeline.scrapper.api.get(
			params=tpl,
			headers=header
		)
		print(api_result)
		wikitex = pipeline.scrapper.obtain_wikitex(api_result)
		data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
		infobox = pipeline.parser.infobox(wikitex)
		is_d = False
		is_deck = False
		if (block in ["D", "DZ"]):
			is_d = True
		rows = pipeline.storage.prepare_data(data, 6, is_d=is_d, is_deck=is_deck, infobox=infobox)
		df = pd.DataFrame(rows, columns=[
			"CardNo", "Name", "Grade", "Faction",
			"FactionType", "Type", "Rarity", "Release"
		])
		set_number = pipeline.classifier.obtain_set_number(
			data[0]
		)
		path = build_set_path(
			category="boosters",
			set_type="main",
			block=block,
			set_number=set_number
		)
		path.parent.mkdir(
			parents=True,
			exist_ok=True
		)
		df.to_parquet(path)

		print(data)

In [ ]:
db = pd.read_parquet("data/database/boosters/booster sets/d/set_08.parquet")
db.loc[db["CardNo"] == "None"]

In [ ]:
db[90:100]

In [ ]:
db[119:210]

In [ ]:
pipeline.storage.d

In [ ]:
dz_consults = pipeline.parser.make_consults(pipeline.storage.d)

In [ ]:
dz_consults[7]

In [ ]:
api_answer = await pipeline.scrapper.api.get(params=dz_consults[7], headers=header)

In [ ]:
wikitex = pipeline.scrapper.obtain_wikitex(api_answer)
data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
infobox = pipeline.parser.infobox(wikitex)

In [ ]:
infobox['release date']

In [ ]:
data[96].params

In [ ]:
if data[96].params[3] in NATIONS:
	print("XD")

In [ ]:
y

In [ ]:
data = pipeline.storage.prepare_data([y], 6, infobox=infobox)

In [ ]:
df = pd.DataFrame(data, columns=[
	"CardNo", "Name", "Grade", "Faction",
	"FactionType", "Type", "Rarity", "Release"
])

In [ ]:
df

In [ ]:
await main_scrap_routine(pipeline.parser, pipeline.storage, pipeline.scrapper, pipeline.classifier)

In [ ]:
api_answer["query"]["pages"]["525299"]["revisions"][0]["slots"]["main"]["*"]

In [ ]:
api_result = await pipeline.scrapper.api.get(
	params=dz_consults[0],
	headers=header
)

In [ ]:
api_result

In [ ]:
api_answer = await pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
wikitex = pipeline.scrapper.obtain_wikitex(api_answer)
data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
infobox = pipeline.parser.infobox(wikitex)
rows = pipeline.storage.prepare_data(data, 5, is_d=True, infobox=infobox)
df = pd.DataFrame(rows, columns=[
	"CardNo", "Name", "Grade", "Faction",
	"FactionType", "Type", "Rarity", "Release"
])
df.to_parquet("data/database/wrong.parquet")

In [ ]:
df = pd.read_parquet("data/database/boosters/booster sets/v/set_-1_8.parquet")

In [ ]:
df

In [ ]:
curl = pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
content = pipeline.scrapper.obtain_links(curl)
pipeline.parser.clean_trash_from_set(crude_consults, content, 4)
card_consult = pipeline.parser.make_consults("consult", content)
main_table = pipeline.scrapper.api.get(params=card_consult[0], headers=header)